# 03 · BERTopic Modeling

Inductively extract **media frames** with BERTopic. Hyper-parameters come from
`config.yaml` (`modeling` section). For a small corpus, consider lowering
`umap.n_components` to 2–3 and `hdbscan.min_cluster_size` to 3.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src.config import load_config
from src import modeling

config = load_config('../config.yaml')
df = pd.read_csv(config['paths']['processed'] / 'articles_all.csv', parse_dates=['date'])
print(len(df), 'articles')

In [ ]:
# Fit BERTopic (downloads the embedding model on first run).
model, topics, probs = modeling.fit_model(config, df)
summary = modeling.topic_summary(model)
summary

In [ ]:
# Persist model + summary for the analysis notebook.
res = config['paths']['results']
summary.to_csv(res / 'topic_summary.csv', index=False)
modeling.save_model(model, res / 'bertopic_model')
df.assign(topic=topics).to_csv(res / 'articles_with_topics.csv', index=False)
print('saved ->', res)

## Step 4 · Manual frame annotation
Read the most representative articles per topic, then name each frame.

In [ ]:
reps = modeling.representative_docs(model, df, topics, top_n=10)
reps.to_csv(res / 'representative_docs.csv', index=False)
reps.head(20)

In [ ]:
# Fill in a human label for each topic and export the annotation table.
frame_names = {
    # topic_id: 'Frame name',  e.g.
    # 0: 'Far-right radicalization engine',
    # 1: 'Meme / internet-culture incubator',
}
annot = summary[summary['topic_id'] != -1].copy()
annot['frame_name'] = annot['topic_id'].map(frame_names)
annot[['topic_id','frame_name','keywords','count']].to_csv(res / 'frames_annotation.csv', index=False)
annot[['topic_id','frame_name','keywords','count']]